In [1]:
import torch
import torch.nn as nn
import numpy as np
torch.__version__

'2.9.1'

3.1 logistic回归实战
在这一章里面，我们将处理一下结构化数据，并使用logistic回归对结构化数据进行简单的分类。
3.1.1 logistic回归介绍
logistic回归是一种广义线性回归（generalized linear model），与多重线性回归分析有很多相同之处。它们的模型形式基本上相同，都具有 wx + b，其中w和b是待求参数，其区别在于他们的因变量不同，多重线性回归直接将wx+b作为因变量，即y =wx+b,而logistic回归则通过函数L将wx+b对应一个隐状态p，p =L(wx+b),然后根据p 与1-p的大小决定因变量的值。如果L是logistic函数，就是logistic回归，如果L是多项式函数就是多项式回归。

说的更通俗一点，就是logistic回归会在线性回归后再加一层logistic函数的调用。

logistic回归主要是进行二分类预测，我们在激活函数时候讲到过 Sigmod函数，Sigmod函数是最常见的logistic函数，因为Sigmod函数的输出的是是对于0~1之间的概率值，当概率大于0.5预测为1，小于0.5预测为0。

下面我们就来使用公开的数据来进行介绍

3.1.2 UCI German Credit  数据集

UCI German Credit是UCI的德国信用数据集，里面有原数据和数值化后的数据。

German Credit数据是根据个人的银行贷款信息和申请客户贷款逾期发生情况来预测贷款违约倾向的数据集，数据集包含24个维度的，1000条数据，

在这里我们直接使用处理好的数值化的数据，作为展示。

[地址](https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/)

3.2 代码实战
我们这里使用的 german.data-numeric是numpy处理好数值化数据，我们直接使用numpy的load方法读取即可

In [2]:
data=np.loadtxt("german.data-numeric")

数据读取完成后我们要对数据做一下归一化的处理

In [3]:
# 对数据集中除最后一列外的所有特征列进行标准化（Z-score 归一化），让每个特征的均值为 0、标准差为 1
# 作用：消除不同特征之间的量纲差异（比如 “收入（元）” 和 “年龄（岁）” 的数值范围差异），让模型训练更稳定、收敛更快。
n,l=data.shape
print(n,l)
for j in range(l-1):
    meanVal=np.mean(data[:,j]) # 计算第j列的均值
    # data[:, j]：取二维数组的第 j 列所有行（即该特征的所有样本值）；
    # 标准差反映特征值的离散程度，标准化会让离散程度统一为 1
    stdVal=np.std(data[:,j]) # 计算第j列的标准差
    # 标准化公式：(特征值 - 列均值) / 列标准差，目的是消除量纲差异；
    data[:,j]=(data[:,j]-meanVal)/stdVal # 标准化

打乱数据

In [4]:
np.random.shuffle(data)

区分训练集和测试集，由于这里没有验证集，所以我们直接使用测试集的准确度作为评判好坏的标准

区分规则：900条用于训练，100条作为测试

german.data-numeric的格式为，前24列为24个维度，最后一个为要打的标签（0，1），所以我们将数据和标签一起区分出来

In [5]:
#行维度 :900：取第 0 行到第 899 行（前 900 个样本）；
# 列维度 :l-1：取第 0 列到第 l-2 列（所有特征列，排除最后一列标签）；
train_data=data[:900,:l-1]
#列维度 l-1：只取最后一列
#原始标签是 1（好信用）/2（坏信用），减 1 后变为 0/1（分类模型通常用 0/1 表示二分类标签）；
train_lab=data[:900,l-1]-1
#取第 900 行到最后一行（后 100 个样本）；
test_data=data[900:,:l-1]
test_lab=data[900:,l-1]-1

下面我们定义模型，模型很简单

In [6]:
class LR(nn.Module):
    def __init__(self):
        super(LR,self).__init__()
        # 2：输出两个值，分别表示 “类别 0 概率” 和 “类别 1 概率”。
        # 也可以输出1类，但这里把二分类当作 “2 类多分类” 来处理，因此输出维度设为 2。
        self.fc=nn.Linear(24,2) # 由于24个维度已经固定了，所以这里写24
    def forward(self,x):
        out=self.fc(x)
        print(out.size())
        out=torch.sigmoid(out)
        print(out.size())
        return out
        

测试集上的准确率

In [25]:
#pred：模型输出的预测概率，维度通常为 (batch_size, 2)（比如 [[0.3, 0.7], [0.8, 0.2]]）；
#pred.max(-1) —— 取最后一维的最大值,返回值：一个元组 (最大值, 最大值的索引)
# pred.max(-1)[1] —— 提取最大值的索引（即预测类别）
def test(pred,lab):
    print(pred.size())
    # pred.max(-1) 返回的是 PyTorch 内置的 torch.return_types.max 元组，包含两个核心部分：values（最大值）和 indices（最大值的索引）
    print('pred.max(-1): ',pred.max(-1))
    t=pred.max(-1)[1]==lab
    # 示例 mean([1.0, 0.0]) = 0.5（准确率 50%）
    return torch.mean(t.float())

下面就是对一些设置

In [16]:
net=LR() 
criterion=nn.CrossEntropyLoss() # 使用CrossEntropyLoss损失
optm=torch.optim.Adam(net.parameters()) # Adam优化
epochs=1 # 训练1000次


下面开始训练了

In [27]:
for i in range(epochs):
    # 指定模型为训练模式，计算梯度
    net.train()
    # 输入值都需要转化成torch的Tensor
    x=torch.from_numpy(train_data).float()
    y=torch.from_numpy(train_lab).long()
    y_hat=net(x)
    loss=criterion(y_hat,y) # 计算损失
    optm.zero_grad() # 前一步的损失清零
    loss.backward() # 反向传播
    optm.step() # 优化
    # if (i+1)%100 ==0 : # 这里我们每100次输出相关的信息
    if i ==0 : # 测试下
        # 指定模型为计算模式
        net.eval()
        test_in=torch.from_numpy(test_data).float()
        test_l=torch.from_numpy(test_lab).long()
        test_out=net(test_in)
        # 使用我们的测试函数计算准确率
        accu=test(test_out,test_l)
        print("Epoch:{},Loss:{:.4f},Accuracy：{:.2f}".format(i+1,loss.item(),accu))

torch.Size([100, 2])
pred.max(-1):  torch.return_types.max(
values=tensor([0.5993, 0.5975, 0.4864, 0.6325, 0.4927, 0.4839, 0.7293, 0.5050, 0.5578,
        0.5674, 0.5834, 0.5370, 0.6067, 0.7232, 0.6008, 0.4485, 0.6178, 0.3496,
        0.7702, 0.6951, 0.6921, 0.6416, 0.5538, 0.4725, 0.5015, 0.5833, 0.6197,
        0.5367, 0.7890, 0.6610, 0.6473, 0.6700, 0.4704, 0.5704, 0.5880, 0.5038,
        0.4845, 0.3894, 0.4146, 0.4426, 0.8097, 0.4507, 0.3784, 0.4144, 0.4861,
        0.5011, 0.5949, 0.5930, 0.5525, 0.6545, 0.5228, 0.6764, 0.8256, 0.6669,
        0.6030, 0.7519, 0.4978, 0.5722, 0.6171, 0.4769, 0.6383, 0.5339, 0.5115,
        0.6419, 0.5014, 0.7402, 0.6596, 0.4042, 0.3664, 0.6068, 0.5703, 0.5795,
        0.6310, 0.6158, 0.6169, 0.5284, 0.5237, 0.5728, 0.4490, 0.5320, 0.5438,
        0.5280, 0.6406, 0.5103, 0.4453, 0.5377, 0.4321, 0.5991, 0.6531, 0.4970,
        0.4375, 0.4159, 0.6465, 0.6415, 0.3907, 0.4609, 0.4568, 0.4996, 0.6675,
        0.4683], grad_fn=<MaxBackward0>),
indices=ten

训练完成了，我们的准确度达到了80%